# 🐝 Lab 4.2 — Class-Conditional Image Generation with Classifier-Free Guidance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab11_diffusion_mnist_cfg.ipynb)

⏱ **Time:** ~2 hours  |  📅 **Day 4 — Generative Worlds**  |  ⚡ **GPU recommended** (Runtime → Change runtime type → T4). CPU works too — you'll get blurrier digits.

🎯 **Learning objectives** — by the end of this lab you will be able to:
- Scale the flow-matching recipe from Lab 4.1 from 2D points to **images**, with a small time- and class-conditioned **UNet**.
- Visualize the **forward noising** path and the reverse **denoising filmstrip**, including the model's running guess of the clean image.
- Train with **label dropout** and sample with **classifier-free guidance (CFG)**; measure the **quality ↔ diversity tradeoff** as the guidance scale grows.
- Explain and implement the core of a **Diffusion Transformer (DiT)**: patchify, transformer blocks, and **adaLN-Zero** conditioning.
- Describe, at a conceptual level, **latent diffusion**, **few-step distillation / consistency models**, and how 2025–2026 **video and world models** build on the same ideas.

🧰 **Prerequisites:** Lab 4.1 (flow matching, Euler sampling), convolutional networks, PyTorch training loops.

**Why this matters in 2026.** Text-to-image and text-to-video systems are conditional generators: "draw *this*". The single most important knob at sampling time is the **guidance scale** — the reason prompts are followed faithfully, and the reason over-guided images look saturated and samey. Today you'll build a complete conditional generator, end to end, small enough to train during a coffee break, and turn that knob yourself. 🍯

### 🗺️ Roadmap

| Part | What you'll do | ⏱ min |
|---|---|---|
| Setup | Imports, seeds, plotting helpers | 5 |
| 1 | Load MNIST and visualize the forward "noising" path | 10 |
| 2 | Build a tiny time- and class-conditioned UNet | 15 |
| 3 | Add label dropout and train (runs ~6 min on CPU, ~10 min on GPU; read ahead meanwhile!) | 20 |
| 4 | Sample with classifier-free guidance: filmstrip, **animated denoising grid**, guidance sweep, fidelity vs diversity | 30 |
| 5 | Stretch: build a tiny Diffusion Transformer (DiT) with adaLN-Zero | 20 |
| 6 | From MNIST to 2026: latents, few-step samplers, video and world models | 15 |
| Wrap-up | Boss Level, key takeaways, interview corner | 5 |

In [ ]:
# 🛠️ Setup — torch, torchvision, matplotlib and sklearn are preinstalled on Colab
import os, math, time, random, copy
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode (tiny runs). Normal users: leave it off.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(42)

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA", "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})

import contextlib, io

@contextlib.contextmanager
def quiet():
    """Swallow download progress bars / chatter (errors still raise)."""
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        yield

def grid_canvas(imgs, nrow):
    """(N, 1, H, W) in [-1, 1] -> one (rows*H, cols*W) image in [0, 1] with thin white borders."""
    imgs = imgs.detach().cpu().clamp(-1, 1)
    n, _, h, w = imgs.shape
    ncol = int(math.ceil(n / nrow))
    canvas = torch.ones(nrow * (h + 2), ncol * (w + 2))
    for i in range(n):
        r, c = i // ncol, i % ncol
        canvas[r * (h + 2) + 1: r * (h + 2) + 1 + h, c * (w + 2) + 1: c * (w + 2) + 1 + w] = (imgs[i, 0] + 1) / 2
    return canvas

def show_grid(imgs, nrow, title=None, ax=None, scale=1.0):
    """imgs: (N, 1, H, W) in [-1, 1]."""
    canvas = grid_canvas(imgs, nrow)
    ncol = int(math.ceil(imgs.shape[0] / nrow))
    if ax is None:
        _, ax = plt.subplots(figsize=(scale * ncol * 0.45 + 0.5, scale * nrow * 0.45 + 0.5))
    ax.imshow(canvas, cmap="gray", vmin=0, vmax=1); ax.axis("off")
    if title: ax.set_title(title)
    return ax

print(f"torch {torch.__version__} | device = {DEVICE} | FAST = {FAST}")

## Part 1 — Data, and the forward "noising" path 🌫️

We use **MNIST** (switch `DATASET = "fashion"` for FashionMNIST). Pixels are scaled to $[-1, 1]$ so that data and Gaussian noise live on comparable scales.

We keep the Lab 4.1 conventions: **$t=0$ is pure noise, $t=1$ is clean data**, and the straight path
$$x_t = (1-t)\,\varepsilon + t\,x_1,\qquad \varepsilon \sim \mathcal{N}(0, I),\qquad \frac{dx_t}{dt} = x_1 - \varepsilon .$$
The whole dataset (60k × 28 × 28) is just 188 MB as float32, so we keep it in one tensor and sample random minibatches — no `DataLoader` needed.

In [ ]:
from torchvision import datasets
DATASET = "mnist"          # or "fashion"
DATA_ROOT = os.environ.get("LAB_DATA", "./data")

try:
    ds_cls = datasets.MNIST if DATASET == "mnist" else datasets.FashionMNIST
    with quiet():                                               # hide the download progress bar
        train_ds = ds_cls(DATA_ROOT, train=True, download=True)
    X = train_ds.data.float().div(255).unsqueeze(1)          # (60000, 1, 28, 28) in [0, 1]
    Y = train_ds.targets.clone().long()
    CLASS_NAMES = [str(i) for i in range(10)] if DATASET == "mnist" else \
        ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]
except Exception as e:                                          # offline fallback: sklearn's 8x8 digits, upsampled
    print(f"⚠️ torchvision download failed ({type(e).__name__}); falling back to sklearn digits")
    from sklearn.datasets import load_digits
    d = load_digits()
    X = F.interpolate(torch.tensor(d.images).float().div(16).unsqueeze(1), size=28, mode="bilinear", align_corners=False)
    Y = torch.tensor(d.target).long()
    CLASS_NAMES = [str(i) for i in range(10)]

if FAST:
    X, Y = X[:4000], Y[:4000]
X = (X * 2 - 1).to(DEVICE)                                      # [-1, 1]
Y = Y.to(DEVICE)
N_CLASSES, NULL = 10, 10                                       # label id 10 = "no label" (used for CFG)
print("data:", tuple(X.shape), "| classes:", CLASS_NAMES)
show_grid(X[:40], nrow=4, title="A few real training images");

### 🧪 Your Turn #1 — the forward path for images

Write `add_noise(x1, eps, t)` returning $x_t = (1-t)\,\varepsilon + t\,x_1$ for image batches of shape `(B, 1, 28, 28)` and `t` of shape `(B,)`. *Hint:* `t.view(-1, 1, 1, 1)`.

In [ ]:
def add_noise(x1, eps, t):
    # TODO: broadcast t over (C, H, W) and return the straight-line interpolation
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
def add_noise(x1, eps, t):
    t = t.view(-1, 1, 1, 1)
    return (1 - t) * eps + t * x1

In [ ]:
# ✔️ Checkpoint
_x, _e = torch.randn(4, 1, 28, 28), torch.randn(4, 1, 28, 28)
assert torch.allclose(add_noise(_x, _e, torch.ones(4)), _x) and torch.allclose(add_noise(_x, _e, torch.zeros(4)), _e)
assert torch.allclose(add_noise(_x, _e, torch.tensor([0.0, 0.25, 0.5, 1.0]))[1], 0.75 * _e[1] + 0.25 * _x[1])
print("✔️ add_noise works.")

ts_show = torch.tensor([1.0, 0.9, 0.75, 0.6, 0.45, 0.3, 0.15, 0.0])
x_demo, eps_demo = X[:6], torch.randn_like(X[:6])
frames = torch.stack([add_noise(x_demo, eps_demo, t.expand(6).to(DEVICE)) for t in ts_show], 1)   # (6, 8, 1, 28, 28)
fig, ax = plt.subplots(figsize=(8, 6.5))
show_grid(frames.reshape(-1, 1, 28, 28), nrow=6, ax=ax, title="Forward path: t = " + ", ".join(f"{t:.2f}" for t in ts_show.tolist()) + "  (data → noise)")
plt.show()

Notice that by $t \approx 0.3$ a human can barely see the digit, yet the model must still produce a *direction* there. Most of the "creative" decisions (which digit shape, which stroke style) are made at low $t$; the last steps (high $t$) just sharpen details. Keep this in mind when we look at the filmstrip and at guidance.

> **🍯 Key takeaway:** Images use exactly the same straight path as 2D points, $x_t = (1-t)\varepsilon + t\,x_1$. The digit becomes unreadable surprisingly early as $t$ falls, so the model makes its big decisions at high noise.

## Part 2 — A tiny conditional UNet 🏗️

**Why a UNet (Ronneberger et al., 2015)?** The velocity at each pixel depends on local stroke structure *and* on global layout. A UNet downsamples to see the big picture, upsamples back to full resolution, and uses **skip connections** to carry fine detail across.

Our CPU-friendly design:
- **Space-to-depth stem:** `pixel_unshuffle` turns each 2×2 pixel block into 4 channels (28×28×1 → 14×14×4), losslessly. This cuts compute ~4× and is the same "patchify" idea DiT uses (Part 5).
- **Residual blocks** with GroupNorm + SiLU; resolutions 14×14 → 7×7 → 14×14.
- **Conditioning:** a sinusoidal **time embedding** plus a learned **class embedding** with 11 entries (10 digits + a **null** token). The sum modulates each block via a per-channel **scale and shift** (FiLM; Perez et al., 2018) — a cousin of DiT's adaLN.
- The output layer is **zero-initialized** so the untrained model predicts $v=0$, a calm starting point.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim, scale=1000.0):
        super().__init__()
        half = dim // 2
        self.register_buffer("freqs", torch.exp(-math.log(10000.0) * torch.arange(half) / half))
        self.scale = scale
    def forward(self, t):
        ang = self.scale * t[:, None] * self.freqs[None]
        return torch.cat([ang.sin(), ang.cos()], -1)

class ResBlock(nn.Module):
    def __init__(self, cin, cout, emb_dim):
        super().__init__()
        self.n1, self.c1 = nn.GroupNorm(8, cin), nn.Conv2d(cin, cout, 3, padding=1)
        self.film = nn.Linear(emb_dim, 2 * cout)
        self.n2, self.c2 = nn.GroupNorm(8, cout), nn.Conv2d(cout, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x, emb):
        h = self.c1(F.silu(self.n1(x)))
        scale, shift = self.film(emb)[:, :, None, None].chunk(2, dim=1)
        h = self.n2(h) * (1 + scale) + shift                     # FiLM conditioning
        h = self.c2(F.silu(h))
        return h + self.skip(x)

class TinyUNet(nn.Module):
    def __init__(self, ch=64, n_classes=10, emb_dim=128):
        super().__init__()
        self.temb = nn.Sequential(SinusoidalTimeEmbedding(emb_dim), nn.Linear(emb_dim, emb_dim), nn.SiLU(), nn.Linear(emb_dim, emb_dim))
        self.cemb = nn.Embedding(n_classes + 1, emb_dim)          # +1 for the NULL label
        self.inc = nn.Conv2d(4, ch, 3, padding=1)
        self.down1 = ResBlock(ch, ch, emb_dim)
        self.to7 = nn.Conv2d(ch, 2 * ch, 3, stride=2, padding=1)
        self.mid1, self.mid2 = ResBlock(2 * ch, 2 * ch, emb_dim), ResBlock(2 * ch, 2 * ch, emb_dim)
        self.to14 = nn.Conv2d(2 * ch, ch, 3, padding=1)
        self.up1 = ResBlock(2 * ch, ch, emb_dim)
        self.out = nn.Sequential(nn.GroupNorm(8, ch), nn.SiLU(), nn.Conv2d(ch, 4, 3, padding=1))
        nn.init.zeros_(self.out[-1].weight); nn.init.zeros_(self.out[-1].bias)
    def forward(self, x, t, y):
        emb = F.silu(self.temb(t) + self.cemb(y))
        h = self.inc(F.pixel_unshuffle(x, 2))                       # (B, ch, 14, 14)
        s1 = self.down1(h, emb)
        h = self.mid2(self.mid1(self.to7(s1), emb), emb)            # (B, 2ch, 7, 7)
        h = self.to14(F.interpolate(h, scale_factor=2, mode="nearest"))
        h = self.up1(torch.cat([h, s1], dim=1), emb)                # skip connection
        return F.pixel_shuffle(self.out(h), 2)                      # back to (B, 1, 28, 28)

_u = TinyUNet(ch=32)
_o = _u(torch.randn(2, 1, 28, 28), torch.rand(2), torch.tensor([3, NULL]))
print("output:", tuple(_o.shape), "| all zeros at init:", bool((_o == 0).all()))

> **🍯 Key takeaway:** A UNet sees both local strokes and the global layout. Injecting $(t, y)$ as a FiLM scale and shift lets *one* network handle every noise level and every class.

## Part 3 — Training with label dropout 🎲

**Classifier-free guidance (Ho & Salimans, 2022)** needs both a conditional model $v_\theta(x,t,y)$ and an unconditional one $v_\theta(x,t,\varnothing)$. The trick: train **one** network for both by replacing the label with the null token with probability $p_{\text{uncond}}$ (≈10–20%):
$$\mathcal{L} = \mathbb{E}_{t,\varepsilon,(x_1,y)}\Big\|v_\theta\big(x_t, t, \tilde y\big) - (x_1 - \varepsilon)\Big\|^2,\qquad \tilde y = \begin{cases}\varnothing & \text{w.p. } p_{\text{uncond}}\\ y & \text{otherwise.}\end{cases}$$

### 🧪 Your Turn #2 — label dropout

Implement `drop_labels(y, p, null_id)`: return a **new** tensor where each entry is independently replaced by `null_id` with probability `p`. Don't modify `y` in place!

In [ ]:
def drop_labels(y, p, null_id=NULL):
    # TODO: mask = (uniform random numbers of y's shape) < p ; return y with masked entries set to null_id
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
def drop_labels(y, p, null_id=NULL):
    mask = torch.rand(y.shape, device=y.device) < p
    return torch.where(mask, torch.full_like(y, null_id), y)

In [ ]:
# ✔️ Checkpoint
_y = torch.randint(0, 10, (20000,))
_yd = drop_labels(_y, 0.1)
assert _yd is not _y and (_y != NULL).all(), "do not modify the input in place"
_frac = (_yd == NULL).float().mean().item()
assert abs(_frac - 0.1) < 0.01, f"expected ~10% dropped, got {_frac:.3f}"
assert (_yd[_yd != NULL] == _y[_yd != NULL]).all(), "kept labels must be unchanged"
assert (drop_labels(_y, 0.0) == _y).all() and (drop_labels(_y, 1.0) == NULL).all()
print(f"✔️ drop_labels ok — dropped {_frac:.1%}")

### The training loop (time-budgeted ⏱)

To keep runtimes predictable on any hardware, training runs until **either** a step limit **or** a wall-clock budget is reached. We also keep an **EMA** (exponential moving average) copy of the weights — a nearly free trick that noticeably improves sample quality in diffusion/flow models (we sample from the EMA model).

| Hardware | Default budget | What to expect |
|---|---|---|
| CPU | ~6 min | blurry but recognizable digits |
| T4 GPU | ~10 min | clean digits |

In [ ]:
P_UNCOND = 0.15
if DEVICE == "cuda":
    CH, BATCH, LR, MAX_STEPS, BUDGET_MIN = 64, 256, 2e-3, 30000, 10.0
else:
    CH, BATCH, LR, MAX_STEPS, BUDGET_MIN = 32, 64, 2e-3, 30000, 6.0
if FAST:
    CH, BATCH, MAX_STEPS, BUDGET_MIN = 16, 32, 40, 1.0

def fm_loss(model, x1, y):
    eps = torch.randn_like(x1)
    t = torch.rand(x1.shape[0], device=x1.device)
    xt = add_noise(x1, eps, t)
    return F.mse_loss(model(xt, t, drop_labels(y, P_UNCOND)), x1 - eps)

@torch.no_grad()
def ema_update(ema, model, decay):
    for pe, pm in zip(ema.parameters(), model.parameters()):
        pe.lerp_(pm, 1 - decay)

def train(model, loss_fn=fm_loss, max_steps=MAX_STEPS, budget_min=BUDGET_MIN, batch=BATCH, lr=LR, ema_max=0.999, tag="model"):
    seed_everything(0)
    model = model.to(DEVICE).train()
    ema = copy.deepcopy(model).eval().requires_grad_(False)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    losses, t0, step = [], time.time(), 0
    while step < max_steps and (time.time() - t0) < budget_min * 60:
        idx = torch.randint(0, X.shape[0], (batch,), device=DEVICE)
        for g in opt.param_groups:
            g["lr"] = lr * min(1.0, (step + 1) / 200)           # short warmup
        loss = loss_fn(model, X[idx], Y[idx])
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        ema_update(ema, model, min(ema_max, (1 + step) / (10 + step)))
        losses.append(loss.item()); step += 1
        if step % 500 == 0:
            print(f"  [{tag}] step {step:5d} | loss {np.mean(losses[-500:]):.4f} | {time.time() - t0:5.0f}s")
    print(f"✅ [{tag}] trained {step} steps ({step * batch / X.shape[0]:.1f} epochs) in {(time.time() - t0) / 60:.1f} min")
    return ema, losses

unet = TinyUNet(ch=CH)
print(f"TinyUNet: {sum(p.numel() for p in unet.parameters()) / 1e6:.2f}M params | batch {BATCH} | budget {BUDGET_MIN} min on {DEVICE}")
ema_unet, unet_losses = train(unet, tag="unet")

plt.figure(figsize=(6, 3))
plt.plot(np.convolve(unet_losses, np.ones(25) / 25, mode="valid"))
plt.xlabel("step"); plt.ylabel("flow-matching loss"); plt.title("Training loss (smoothed)"); plt.show()

> **🍯 Key takeaway:** Label dropout trains a conditional and an unconditional model in one set of weights, which is the prerequisite for classifier-free guidance. EMA weights are a nearly free quality boost.

## Part 4 — Sampling with classifier-free guidance 🎚️

**The idea.** By Bayes, $\nabla \log p(x\mid y) = \nabla \log p(x) + \nabla \log p(y \mid x)$. Classifier guidance (Dhariwal & Nichol, 2021) adds an extra push $w\,\nabla \log p(y\mid x)$ from a separately trained noisy classifier. **Classifier-free** guidance gets the "classifier direction" for free as the *difference* between the conditional and unconditional predictions, and extrapolates along it:
$$\tilde v(x,t,y) = v_\theta(x,t,\varnothing) + w\,\big[v_\theta(x,t,y) - v_\theta(x,t,\varnothing)\big].$$
- $w=0$: unconditional model (ignores $y$) · $w=1$: plain conditional model · $w>1$: **over-emphasize** what makes the sample look like class $y$.

(Ho & Salimans write it as $(1+w')\,\varepsilon_c - w'\,\varepsilon_u$, i.e. $w = 1 + w'$; check the convention whenever you read "guidance scale 7.5"!) Cost: **two network evaluations per step** — we batch them together.

### 🧪 Your Turn #3 — the CFG combination

In [ ]:
def cfg_velocity(v_cond, v_uncond, w):
    # TODO: return the guided velocity
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
def cfg_velocity(v_cond, v_uncond, w):
    return v_uncond + w * (v_cond - v_uncond)

In [ ]:
# ✔️ Checkpoint
_vc, _vu = torch.randn(3, 1, 28, 28), torch.randn(3, 1, 28, 28)
assert torch.allclose(cfg_velocity(_vc, _vu, 0.0), _vu, atol=1e-6), "w=0 must be unconditional"
assert torch.allclose(cfg_velocity(_vc, _vu, 1.0), _vc, atol=1e-6), "w=1 must be the plain conditional model"
assert torch.allclose(cfg_velocity(_vc, _vu, 3.0), 3 * _vc - 2 * _vu, atol=1e-6)
print("✔️ CFG combination correct.")

### 🧪 Your Turn #4 — the model's running guess of the clean image

From the straight path, $x_1 = x_t + (1-t)\,(x_1 - \varepsilon)$, so a velocity prediction implies a **clean-image prediction** $\hat x_1 = x_t + (1-t)\,v$. Implement `predict_clean(xt, v, t)` (`t` has shape `(B,)`). We'll use it for the filmstrip.

In [ ]:
def predict_clean(xt, v, t):
    # TODO
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
def predict_clean(xt, v, t):
    return xt + (1 - t.view(-1, 1, 1, 1)) * v

In [ ]:
# ✔️ Checkpoint — with the *true* per-sample velocity, the clean image is recovered exactly
_x, _e, _t = torch.randn(5, 1, 28, 28), torch.randn(5, 1, 28, 28), torch.rand(5)
assert torch.allclose(predict_clean(add_noise(_x, _e, _t), _x - _e, _t), _x, atol=1e-5)
print("✔️ predict_clean works.")

@torch.no_grad()
def sample(model, y, w=3.0, n_steps=30, seed=0, keep=None):
    """Euler-integrate the CFG velocity from noise (t=0) to data (t=1).
    keep: optional list of step indices at which to record (x_t, x1_hat)."""
    model.eval()
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    y = y.to(DEVICE)
    x = torch.randn(len(y), 1, 28, 28, device=DEVICE, generator=g)
    ts = torch.linspace(0, 1, n_steps + 1, device=DEVICE)
    null = torch.full_like(y, NULL)
    film = []
    for i in range(n_steps):
        t = ts[i].expand(len(y))
        if w == 1.0:
            v = model(x, t, y)
        elif w == 0.0:
            v = model(x, t, null)
        else:   # one batched forward pass for (cond, uncond)
            vc, vu = model(torch.cat([x, x]), torch.cat([t, t]), torch.cat([y, null])).chunk(2)
            v = cfg_velocity(vc, vu, w)
        if keep is not None and i in keep:
            film.append((x.clone(), predict_clean(x, v, t)))
        x = x + (ts[i + 1] - ts[i]) * v
    return (x, film) if keep is not None else x

### Denoising filmstrip 🎞️

Top row of each pair: the current state $x_t$. Bottom row: the model's guess $\hat x_1$ at that moment. Watch the guess start as a blurry "average digit of that class" and commit to a specific style as $t \to 1$.

In [ ]:
N_STEPS = 30
keep = [0, 3, 6, 9, 12, 15, 20, 25, 29]
y_film = torch.arange(6)
x_final, film = sample(ema_unet, y_film, w=3.0, n_steps=N_STEPS, seed=1, keep=keep)
fig, axes = plt.subplots(2 * len(y_film), len(keep), figsize=(1.2 * len(keep), 2.4 * len(y_film) * 0.55))
for c, (xt, xh) in enumerate(film):
    for r in range(len(y_film)):
        for k, img in enumerate([xt[r], xh[r]]):
            ax = axes[2 * r + k, c]; ax.imshow(img[0].cpu().clamp(-1, 1), cmap="gray", vmin=-1, vmax=1); ax.set_xticks([]); ax.set_yticks([])
            if c == 0: ax.set_ylabel(("$x_t$ " if k == 0 else "$\\hat{x}_1$ ") + CLASS_NAMES[r], fontsize=8)
    axes[0, c].set_title(f"t={keep[c] / N_STEPS:.2f}", fontsize=9)
plt.suptitle("Denoising filmstrip (w = 3)", y=1.0); plt.tight_layout(); plt.show()

### 🎬 The wow moment: 60 digits emerging from static

One sampling run, every intermediate step recorded. Each row asks for a different digit (w = 3). Press ▶ and watch strokes condense out of pure noise.

In [ ]:
from matplotlib import animation
from IPython.display import HTML
plt.rcParams["animation.embed_limit"] = 30          # MB

y_anim = torch.arange(N_CLASSES).repeat_interleave(6)
KEEP = list(range(0, N_STEPS, 2))                    # every other step keeps the embedded animation small (~1-2 MB)
x_end, film_all = sample(ema_unet, y_anim, w=3.0, n_steps=N_STEPS, seed=3, keep=KEEP)
frames = [grid_canvas(xt, nrow=N_CLASSES) for xt, _ in film_all] + [grid_canvas(x_end, nrow=N_CLASSES)] * 3
frame_t = [k / N_STEPS for k in KEEP] + [1.0] * 3

fig, ax = plt.subplots(figsize=(3.4, 5.4), dpi=72)
im = ax.imshow(frames[0], cmap="gray", vmin=0, vmax=1); ax.axis("off")
ttl = ax.set_title("t = 0.00")
def update(k):
    im.set_data(frames[k]); ttl.set_text(f"t = {frame_t[k]:.2f}")
    return im, ttl
anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=180)
plt.close(fig)
HTML(anim.to_jshtml())

### 🔮 Predict-then-run: the guidance sweep

We'll draw the same noise for every row and sweep $w \in \{0, 1, 2, 3, 5, 8\}$. Before running, predict:
1. At $w=0$, will the rows (one per class label) contain the right digits?
2. As $w$ grows, what happens to the **variety** of styles within a row? To stroke thickness and contrast?
3. Is there a $w$ that's "too much"?

In [ ]:
W_LIST = [0.0, 1.0, 2.0, 3.0, 5.0, 8.0]
PER_CLASS = 6
y_grid = torch.arange(N_CLASSES).repeat_interleave(PER_CLASS)
fig, axes = plt.subplots(2, 3, figsize=(13, 9.5))
for ax, w in zip(axes.flat, W_LIST):
    show_grid(sample(ema_unet, y_grid, w=w, n_steps=N_STEPS, seed=7), nrow=N_CLASSES, ax=ax, title=f"w = {w:g}")
plt.suptitle("Same noise, increasing guidance (rows = requested class)", y=0.99); plt.tight_layout(); plt.show()

### Measuring the tradeoff 📈

Eyeballing is great; numbers are better. We train a quick **MNIST classifier** as a judge and, for each $w$, record:
- **Label agreement** — fraction of samples the judge classifies as the requested class (a crude "precision/fidelity" proxy, in the spirit of Inception-Score-style metrics),
- **Confidence** — mean max-softmax probability,
- **Within-class diversity** — mean pairwise $L_2$ distance between samples requested with the same label.

(Real papers use FID, precision/recall and CLIP-score; the qualitative tradeoff is the same.)

In [ ]:
class Judge(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10))
    def forward(self, x): return self.net(x)

seed_everything(0)
judge = Judge().to(DEVICE); jopt = torch.optim.Adam(judge.parameters(), 1e-3)
for step in range(40 if FAST else 600):
    idx = torch.randint(0, X.shape[0], (128,), device=DEVICE)
    loss = F.cross_entropy(judge(X[idx]), Y[idx]); jopt.zero_grad(); loss.backward(); jopt.step()
judge.eval()
with torch.no_grad():
    idx = torch.arange(min(5000, X.shape[0]), device=DEVICE)
    print(f"judge accuracy on (training) images: {(judge(X[idx]).argmax(1) == Y[idx]).float().mean():.3f}")

N_PER = 4 if FAST else (40 if DEVICE == "cuda" else 16)
y_eval = torch.arange(N_CLASSES).repeat_interleave(N_PER).to(DEVICE)
stats = {"agree": [], "conf": [], "div": []}
for w in W_LIST:
    xs = sample(ema_unet, y_eval, w=w, n_steps=20, seed=11)
    with torch.no_grad():
        p = judge(xs.clamp(-1, 1)).softmax(1)
    stats["agree"].append((p.argmax(1) == y_eval).float().mean().item())
    stats["conf"].append(p.max(1).values.mean().item())
    flat = xs.view(N_CLASSES, N_PER, -1)
    d = torch.cdist(flat, flat)                                         # (classes, N_PER, N_PER)
    stats["div"].append((d.sum((1, 2)) / (N_PER * (N_PER - 1))).mean().item())
    print(f"w = {w:3.0f} | agreement {stats['agree'][-1]:.2f} | confidence {stats['conf'][-1]:.2f} | diversity {stats['div'][-1]:.2f}")

fig, ax1 = plt.subplots(figsize=(6.5, 4))
ax1.plot(W_LIST, stats["agree"], "o-", c=HIVE[3], label="label agreement")
ax1.plot(W_LIST, stats["conf"], "s--", c=HIVE[4], label="judge confidence")
ax1.set_xlabel("guidance scale w"); ax1.set_ylabel("fidelity ↑"); ax1.set_ylim(0, 1.05)
ax2 = ax1.twinx(); ax2.plot(W_LIST, stats["div"], "^-", c=HIVE[2], label="within-class diversity"); ax2.set_ylabel("diversity (pixel L2)", color=HIVE[2])
ax2.spines["right"].set_visible(True)
fig.legend(loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.08), fontsize=9); plt.title("The CFG tradeoff: fidelity ↑, diversity ↓"); plt.show()

**What you should see:** at $w=0$ agreement is near chance (~10%) — the unconditional model ignores the label. Agreement and confidence jump at $w=1$ and keep rising, while diversity falls: guided samples become thicker, higher-contrast "prototypical" digits. Push $w$ far beyond the sweep (try 15 or 30) and Euler integration of the extrapolated field starts to overshoot, giving saturated, blotchy strokes; on big text-to-image models this shows up much earlier as over-saturated, low-variety images. Practical text-to-image systems pick a moderate $w$ (and tricks like guidance only in a limited interval of $t$ — Kynkäänniemi et al., 2024 — reduce the diversity loss).

> **🍯 Key takeaway:** CFG extrapolates from the unconditional prediction toward the conditional one. Raising $w$ buys fidelity and prompt-following at the cost of diversity, and every step costs two network calls.

## Part 5 — Stretch: a tiny Diffusion Transformer (DiT) ⚡ GPU recommended

**DiT (Peebles & Xie, 2023)** replaced the UNet with a plain Vision Transformer and found that sample quality improves predictably as you scale transformer compute (Gflops). The recipe:

1. **Patchify** the (latent) image into $p\times p$ patches → a sequence of tokens (+ positional embeddings).
2. A stack of transformer blocks. Conditioning (timestep + class) enters through **adaLN-Zero**: an MLP on the conditioning vector $c$ outputs per-block *shift* $\beta$, *scale* $\gamma$ and *gate* $\alpha$:
$$h = x + \alpha_1 \odot \mathrm{Attn}\big(\mathrm{LN}(x)(1+\gamma_1) + \beta_1\big),\qquad x' = h + \alpha_2 \odot \mathrm{MLP}\big(\mathrm{LN}(h)(1+\gamma_2) + \beta_2\big).$$
   The MLP producing $(\beta,\gamma,\alpha)$ is **zero-initialized**, so every block starts as the identity — stable at depth.
3. A final adaLN + linear layer maps each token back to a patch; **unpatchify**.

DiT-style transformers (with variations such as joint text-image attention in SD3's MM-DiT) are the dominant backbone for 2024–2026 image and video generators. Here: 4×4 patches → 49 tokens.

### 🧪 Your Turn #5 — patchify / unpatchify
`patchify(x, p)`: `(B, C, H, W)` → `(B, (H/p)(W/p), C·p·p)`, patches in row-major order. `unpatchify(tokens, p, C, H, W)` inverts it.
*Hint:* `x.unfold` works, but `reshape` + `permute` is cleaner: `(B, C, H/p, p, W/p, p)` → `(B, H/p, W/p, C, p, p)`.

In [ ]:
def patchify(x, p):
    # TODO
    pass

def unpatchify(tokens, p, C, H, W):
    # TODO
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
def patchify(x, p):
    B, C, H, W = x.shape
    x = x.reshape(B, C, H // p, p, W // p, p).permute(0, 2, 4, 1, 3, 5)
    return x.reshape(B, (H // p) * (W // p), C * p * p)

def unpatchify(tokens, p, C, H, W):
    B = tokens.shape[0]
    x = tokens.reshape(B, H // p, W // p, C, p, p).permute(0, 3, 1, 4, 2, 5)
    return x.reshape(B, C, H, W)

In [ ]:
# ✔️ Checkpoint
_x = torch.randn(2, 1, 28, 28)
_tok = patchify(_x, 4)
assert _tok.shape == (2, 49, 16), f"expected (2, 49, 16), got {tuple(_tok.shape)}"
assert torch.equal(_tok[0, 0], _x[0, 0, :4, :4].reshape(-1)), "token 0 should be the top-left 4x4 patch"
assert torch.equal(_tok[0, 1], _x[0, 0, :4, 4:8].reshape(-1)), "token 1 should be the next patch to the right"
assert torch.equal(unpatchify(_tok, 4, 1, 28, 28), _x), "unpatchify must invert patchify"
print("✔️ patchify/unpatchify round-trip exact.")

In [ ]:
class DiTBlock(nn.Module):
    def __init__(self, d, heads, mlp_ratio=4):
        super().__init__()
        self.n1 = nn.LayerNorm(d, elementwise_affine=False, eps=1e-6)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.n2 = nn.LayerNorm(d, elementwise_affine=False, eps=1e-6)
        self.mlp = nn.Sequential(nn.Linear(d, mlp_ratio * d), nn.GELU(approximate="tanh"), nn.Linear(mlp_ratio * d, d))
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(d, 6 * d))
        nn.init.zeros_(self.ada[-1].weight); nn.init.zeros_(self.ada[-1].bias)    # adaLN-Zero
    def forward(self, x, c):
        b1, g1, a1, b2, g2, a2 = self.ada(c)[:, None, :].chunk(6, dim=-1)
        h = self.n1(x) * (1 + g1) + b1
        x = x + a1 * self.attn(h, h, h, need_weights=False)[0]
        h = self.n2(x) * (1 + g2) + b2
        return x + a2 * self.mlp(h)

class TinyDiT(nn.Module):
    def __init__(self, patch=4, d=128, depth=4, heads=4, n_classes=10, img=28, C=1):
        super().__init__()
        self.p, self.C, self.img = patch, C, img
        n_tok = (img // patch) ** 2
        self.embed = nn.Linear(C * patch * patch, d)
        self.pos = nn.Parameter(torch.randn(1, n_tok, d) * 0.02)
        self.temb = nn.Sequential(SinusoidalTimeEmbedding(d), nn.Linear(d, d), nn.SiLU(), nn.Linear(d, d))
        self.cemb = nn.Embedding(n_classes + 1, d)
        self.blocks = nn.ModuleList([DiTBlock(d, heads) for _ in range(depth)])
        self.n_out = nn.LayerNorm(d, elementwise_affine=False, eps=1e-6)
        self.ada_out = nn.Sequential(nn.SiLU(), nn.Linear(d, 2 * d))
        self.head = nn.Linear(d, C * patch * patch)
        for m in [self.ada_out[-1], self.head]:
            nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x, t, y):
        c = self.temb(t) + self.cemb(y)
        h = self.embed(patchify(x, self.p)) + self.pos
        for blk in self.blocks:
            h = blk(h, c)
        shift, scale = self.ada_out(c)[:, None, :].chunk(2, dim=-1)
        h = self.head(self.n_out(h) * (1 + scale) + shift)
        return unpatchify(h, self.p, self.C, self.img, self.img)

# ✔️ Checkpoint — adaLN-Zero means every block (and the whole model) starts as identity / zero output
_dit = TinyDiT()
_h, _c = torch.randn(2, 49, 128), torch.randn(2, 128)
assert torch.allclose(_dit.blocks[0](_h, _c), _h), "adaLN-Zero block must be the identity at init"
assert (_dit(torch.randn(2, 1, 28, 28), torch.rand(2), torch.tensor([1, NULL])) == 0).all()
print(f"✔️ TinyDiT: {sum(p.numel() for p in _dit.parameters()) / 1e6:.2f}M params, identity at init.")

In [ ]:
# ⚡ GPU recommended: on CPU we shrink to a short demo run (expect noisy blobs, not digits).
if DEVICE == "cuda":
    DIT_CFG, DIT_BUDGET, DIT_BATCH = dict(d=256, depth=6, heads=8), 8.0, 256
else:
    DIT_CFG, DIT_BUDGET, DIT_BATCH = dict(d=128, depth=4, heads=4), 1.5, 64
if FAST:
    DIT_CFG, DIT_BUDGET, DIT_BATCH = dict(d=64, depth=2, heads=4), 0.2, 16

dit = TinyDiT(**DIT_CFG)
print(f"TinyDiT {DIT_CFG}: {sum(p.numel() for p in dit.parameters()) / 1e6:.2f}M params, budget {DIT_BUDGET} min")
ema_dit, dit_losses = train(dit, budget_min=DIT_BUDGET, batch=DIT_BATCH, lr=1e-3,
                            max_steps=40 if FAST else MAX_STEPS, tag="dit")
show_grid(sample(ema_dit, torch.arange(10).repeat_interleave(4), w=3.0, n_steps=N_STEPS), nrow=10,
          title=f"TinyDiT samples (w=3) after {len(dit_losses)} steps");
plt.show()

**Reflection.** On the same budget the UNet usually wins at this tiny scale: convolutions bake in locality and translation equivariance, which transformers must learn from data. DiT's advantage appears with **scale** — more data, more compute, and latents instead of pixels — where its simple, uniform architecture scales smoothly (Peebles & Xie report FID improving steadily with Gflops).

> **🍯 Key takeaway:** DiT = patchify + transformer + adaLN-Zero. It loses to a UNet at toy scale but scales smoothly, which is why transformer backbones dominate large image and video generators.

## Part 6 — From MNIST to 2026: latents, few-step samplers, video and world models 🌍

Everything you built today survives, almost unchanged, in frontier systems. What changes is *where* the generation happens, *how many steps* it takes, and *what the conditioning is*.

### Latent diffusion
Running a UNet on $1024^2 \times 3$ pixels is expensive. **Latent Diffusion Models** (Rombach et al., 2022 — the basis of Stable Diffusion) first train an autoencoder that compresses images ~8× per side into a latent grid, then run diffusion *in the latent space*, and decode at the end. Text conditioning enters through cross-attention to a text encoder's tokens. Stable Diffusion 3 (Esser et al., 2024) keeps the latent space but switches to a **rectified-flow** objective and a DiT-style backbone (MM-DiT) — i.e. exactly Lab 4.1 + Part 5 of this lab, at scale.

### Few-step sampling: distillation and consistency
Our sampler used 20–30 steps × 2 network calls (CFG). Ways to cut that:
- **Better solvers** for the same model (DDIM; DPM-Solver, Lu et al., 2022; the design-space analysis of Karras et al., 2022).
- **Progressive distillation** (Salimans & Ho, 2022): train a student to match two teacher steps in one, repeat, halving the step count each round.
- **Reflow** (Lab 4.1) straightens paths so few Euler steps suffice; InstaFlow (Liu et al., 2024) combines it with distillation for one-step text-to-image.
- **Consistency models** (Song et al., 2023): learn a function $f_\theta(x_t, t)$ that maps *any* point on a trajectory directly to its endpoint, enforcing self-consistency along the ODE; supports 1–2 step generation. Latent Consistency Models (Luo et al., 2023) apply this to Stable Diffusion.
- **Adversarial distillation** (e.g. ADD / SDXL-Turbo, Sauer et al., 2023): add a GAN-style discriminator loss to get sharp 1–4 step samples.
- **Guidance distillation**: bake the two-pass CFG into a single conditional network.

### Video generation and world models (conceptual)
- **Video diffusion/flow models** treat a clip as a 3D (time × height × width) latent, compressed by a video autoencoder, and denoise all frames jointly so motion stays coherent. OpenAI's Sora technical report (Brooks et al., 2024, *"Video generation models as world simulators"*) describes a diffusion transformer over **spacetime patches** of a video latent — DiT's patchify, extended in time. Open-weight 2024–2025 models whose designs are public (e.g. HunyuanVideo, Wan 2.1) use exactly this recipe — a video VAE, a diffusion transformer, a flow-matching objective, text conditioning and CFG. Closed systems (e.g. Google DeepMind's Veo line) disclose fewer architectural details.
- **World models** additionally condition on **actions**, generating the next frames of an environment interactively. DeepMind's Genie line (Genie, Bruce et al., 2024; Genie 2 and Genie 3 announced in late 2024 and 2025) and NVIDIA's Cosmos world foundation models (2025) aim to provide controllable, physically plausible simulators for games, robotics and autonomous-driving research. Their core loop is still: *encode to latents → predict the future with a large transformer (diffusion/flow or autoregressive) → decode*.
- Open challenges you can now reason about: long-horizon consistency (errors compound autoregressively), physical plausibility (a generative model optimizes likelihood, not physics), compute cost per frame (hence the push to few-step samplers), and evaluation.

> 🐝 *Buzz's caution:* this field moves monthly. Treat product names and capabilities above as a snapshot; for any specific claim, read the primary source.

> **🍯 Key takeaway:** Frontier image, video and world models keep this exact loop and change *where* it runs (latents), *how many steps* it takes (solvers, distillation, consistency) and *what conditions it* (text, actions).

## 🏆 Boss Level

1. **DDPM vs flow matching, head to head.** Write `ddpm_loss(model, x0, y)` with a cosine noise schedule (Nichol & Dhariwal, 2021) and ε-prediction, train the same `TinyUNet` with the same budget, sample with DDIM + CFG on ε, and compare guidance sweeps. Which looks better at 10 steps?
2. **Guidance in a limited interval.** Apply CFG only for $t \in [t_{lo}, t_{hi}]$ (e.g. the middle of the trajectory) and use $w=1$ elsewhere (Kynkäänniemi et al., 2024). Re-run the tradeoff plot: can you keep agreement while recovering diversity?
3. **Latent flow matching.** Train a small convolutional autoencoder to compress 28×28 → 4×7×7, then train the flow model on the latents and decode samples. Compare speed and quality to pixel space.
4. **Two-step digits.** Apply reflow (Lab 4.1) to your MNIST model: generate (noise, sample) pairs with 50 Euler steps, fine-tune on them, and compare 1-, 2- and 4-step samples before/after.

## 🍯 Key Takeaways

- **Same recipe, bigger network:** image flow matching regresses $v_\theta(x_t, t, y)$ onto $x_1 - \varepsilon$ along $x_t = (1-t)\varepsilon + t x_1$, exactly as in Lab 4.1.
- **Small UNets go a long way:** time and class embeddings via FiLM, a zero-initialized output and EMA weights give recognizable digits within minutes, even on a CPU.
- **Label dropout + CFG:** one network learns both $v(x,t,y)$ and $v(x,t,\varnothing)$, and sampling extrapolates $v_\varnothing + w(v_y - v_\varnothing)$.
- **Guidance is a dial, not a free lunch:** higher $w$ means higher fidelity and lower diversity, at 2× compute per step. Too high and samples saturate.
- **Coarse to fine:** the running prediction $\hat x_1 = x_t + (1-t)v$ shows that layout is decided early and details late.
- **DiT** (patchify + transformer + adaLN-Zero) is the scalable backbone behind most large image and video generators.
- **At scale:** generate in a **latent** space, cut steps with **solvers, reflow, distillation or consistency models**, and condition on **text** (or **actions**, for world models).

## 🎤 Explain it to a friend

In one paragraph, explain to a non-technical friend what the **guidance scale** slider in an image generator does. Use the idea of a "generic sketch" versus a "sketch of exactly what you asked for", and explain why turning the slider all the way up makes every image look the same, and a bit cartoonish.

## 💼 Interview corner

**Q1. Explain classifier-free guidance. How is it trained, how is it applied at sampling time, and why does it reduce diversity?**
<details><summary>Answer</summary>

During training, the conditioning label (or text) is replaced with a null token some fraction of the time (≈10–20%), so one network learns both conditional and unconditional predictions. At sampling time, both are evaluated at every step and combined as $\tilde v = v_\varnothing + w\,(v_c - v_\varnothing)$; for scores this approximately follows $\nabla\log p(x) + w\,\nabla\log p(c\mid x)$, which sharpens the implicit classifier term. With $w>1$ it over-weights regions where the condition is most recognizable, which pushes samples toward class "prototypes": higher fidelity and prompt adherence, lower diversity. It also doubles the per-step cost unless the guidance is distilled away.
</details>

**Q2. Why do production systems run diffusion/flow models in a *latent* space instead of pixel space?**
<details><summary>Answer</summary>

Compute scales with the number of spatial positions. A pretrained autoencoder that compresses e.g. 8× per side reduces positions by 64×, and most of the bits it discards are imperceptible high-frequency detail that the decoder can restore. The generative model then spends its capacity on semantics and composition (Rombach et al., 2022). Costs: the autoencoder caps the reconstruction quality, and latents must be well-behaved (regularized) for generation to work well.
</details>

**Q3. What is adaLN-Zero in DiT, and why initialize it to zero?**
<details><summary>Answer</summary>

Each transformer block's LayerNorm scale/shift, plus a residual gate, are predicted from the conditioning vector (timestep + class or text) by a small MLP. That MLP's final layer is zero-initialized, so every residual branch is gated to zero at the start and each block is the identity function. Peebles & Xie found this conditioning scheme gave the best FID among the variants they tried (in-context tokens, cross-attention, adaLN without zero-init), and identity-at-init makes deep stacks train stably.
</details>

## 🐝 Fun fact

MNIST is a remix: its creators combined digits from NIST Special Database 3 (written by U.S. Census Bureau employees) with Special Database 1 (written by high-school students), because the two groups' handwriting differed noticeably. Your diffusion model just learned both styles at once.

## 📚 Further reading

- Ho & Salimans (2022). *Classifier-Free Diffusion Guidance.* arXiv:2207.12598 (NeurIPS 2021 Workshop on Deep Generative Models).
- Dhariwal & Nichol (2021). *Diffusion Models Beat GANs on Image Synthesis.* NeurIPS.
- Ho, Jain & Abbeel (2020). *Denoising Diffusion Probabilistic Models.* NeurIPS.
- Lipman et al. (2023). *Flow Matching for Generative Modeling.* ICLR.
- Peebles & Xie (2023). *Scalable Diffusion Models with Transformers.* ICCV.
- Ronneberger, Fischer & Brox (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation.* MICCAI.
- Perez, Strub, de Vries, Dumoulin & Courville (2018). *FiLM: Visual Reasoning with a General Conditioning Layer.* AAAI.
- Rombach, Blattmann, Lorenz, Esser & Ommer (2022). *High-Resolution Image Synthesis with Latent Diffusion Models.* CVPR.
- Esser et al. (2024). *Scaling Rectified Flow Transformers for High-Resolution Image Synthesis.* ICML.
- Karras, Aittala, Aila & Laine (2022). *Elucidating the Design Space of Diffusion-Based Generative Models.* NeurIPS.
- Salimans & Ho (2022). *Progressive Distillation for Fast Sampling of Diffusion Models.* ICLR.
- Song, Dhariwal, Chen & Sutskever (2023). *Consistency Models.* ICML.
- Nichol & Dhariwal (2021). *Improved Denoising Diffusion Probabilistic Models.* ICML.
- Kynkäänniemi et al. (2024). *Applying Guidance in a Limited Interval Improves Sample and Distribution Quality in Diffusion Models.* NeurIPS.
- Brooks et al. (2024). *Video generation models as world simulators.* OpenAI technical report.
- Bruce et al. (2024). *Genie: Generative Interactive Environments.* ICML.

🐝 *Buzz says: you've now built a conditional image generator from scratch. Next lab: teaching vision and language to speak the same embedding language!*